# Model Optimization: Baseline Evaluation

In this notebook, we'll establish baseline performance metrics for our models using distributed processing. Instead of running the evaluations on our notebook instance, we'll launch separate SageMaker Processing jobs to perform the evaluations on more powerful instances.

## What are Baseline Metrics?

Baseline metrics provide a reference point for measuring the effectiveness of our optimization techniques. By establishing these metrics first, we can quantify the improvements achieved through quantization, pruning, and knowledge distillation.

### Key Metrics We'll Measure:
- **Model Size**: How much storage space the model requires
- **Inference Time**: How long it takes to generate predictions
- **Number of Parameters**: How many trainable parameters the model has

### Distributed Processing Approach
This notebook uses SageMaker Processing jobs to perform evaluations on separate, more powerful instances. This approach allows us to:
1. Use a small, cost-effective instance for our notebook
2. Launch larger instances only when needed for resource-intensive tasks
3. Process multiple models in parallel
4. Scale to larger models that might not fit on the notebook instance

## 1. Import Dependencies

In [ ]:
import osimport jsonimport timeimport pandas as pdimport matplotlib.pyplot as pltimport seaborn as snsimport boto3import sagemakerfrom sagemaker.processing import ProcessingInput, ProcessingOutput, Processorfrom sagemaker.pytorch.processing import PyTorchProcessor# Import our utility functionsfrom optimization_utils import analyze_job_failure, handle_processing_error

## 2. Load Workshop Settings

Load the workshop settings that were configured in the first notebook.

In [ ]:
# Load stored variables
%store -r S3_BUCKET
%store -r AWS_REGION
%store -r SAGEMAKER_ROLE_ARN
%store -r OPTIMIZATION_INSTANCE_TYPE

# Check if variables were successfully retrieved
if 'S3_BUCKET' in locals() and S3_BUCKET != "YOUR_BUCKET_NAME_HERE":
    print("Workshop settings loaded successfully:")
    print(f"S3 Bucket: {S3_BUCKET}")
    print(f"AWS Region: {AWS_REGION}")
    print(f"SageMaker Role ARN: {SAGEMAKER_ROLE_ARN}")
    print(f"Optimization Instance Type: {OPTIMIZATION_INSTANCE_TYPE}")
else:
    print("⚠️ Workshop settings not found or not configured.")
    print("Please run the first notebook (01_introduction_and_setup.ipynb) to configure settings.")
    
    # Set default values that user should update
    S3_BUCKET = "YOUR_BUCKET_NAME_HERE"  # Update this value
    AWS_REGION = "YOUR_REGION_HERE"      # Update this value
    SAGEMAKER_ROLE_ARN = "YOUR_ROLE_ARN_HERE"  # Update this value
    OPTIMIZATION_INSTANCE_TYPE = "ml.c5.xlarge"  # Default optimization instance type
    
    # Store the updated values
    %store S3_BUCKET
    %store AWS_REGION
    %store SAGEMAKER_ROLE_ARN
    %store OPTIMIZATION_INSTANCE_TYPE

## 3. Define Models to Evaluate

We'll define a set of models to evaluate, covering different tasks and model sizes.

In [ ]:
# Define models to evaluate
model_info = {
    "sentiment_analysis": {
        "model_name": "distilbert-base-uncased-finetuned-sst-2-english",
        "task": "sequence-classification"
    },
    "ner": {
        "model_name": "dbmdz/bert-large-cased-finetuned-conll03-english",
        "task": "token-classification"
    },
    "question_answering": {
        "model_name": "distilbert-base-cased-distilled-squad",
        "task": "question-answering"
    },
    "masked_lm": {
        "model_name": "distilroberta-base",
        "task": "masked-lm"
    }
}

# Save model information to a file
with open('model_info.json', 'w') as f:
    json.dump(model_info, f, indent=2)

print(f"Defined {len(model_info)} models for evaluation")

## 4. Define Sample Inputs for Each Task

In [ ]:
# Define sample inputs for each task
sample_inputs = {
    "sentiment_analysis": "I really enjoyed this movie. The acting was superb and the plot was engaging.",
    "ner": "Jeff Bezos founded Amazon in 1994 and the company is headquartered in Seattle, Washington.",
    "question_answering": {
        "question": "What is machine learning?",
        "context": "Machine learning is a branch of artificial intelligence that focuses on building systems that learn from data."
    },
    "masked_lm": "The [MASK] is a large language model trained by OpenAI."
}

## 5. Examine and Upload Baseline Evaluation Script to S3

In this section, we'll examine and upload the Python script that performs the baseline evaluations. This script will be executed on the SageMaker Processing instances.

### What the Script Does:
1. **Loads the model and tokenizer** from Hugging Face
2. **Prepares sample inputs** for inference
3. **Measures performance metrics** like model size and inference time
4. **Saves the model** and metrics to the output directory

The script processes each model defined in the model_info.json file and collects baseline metrics for comparison with optimized models later.

In [ ]:
# Display the baseline evaluation script with syntax highlighting
%pycat baseline_evaluation_script.py

In [ ]:
# Upload the baseline evaluation script to S3
s3_client = boto3.client('s3')
s3_client.upload_file(
    'baseline_evaluation_script.py', 
    S3_BUCKET, 
    'scripts/baseline_evaluation_script.py'
)

# Upload model info to S3
s3_client.upload_file(
    'model_info.json', 
    S3_BUCKET, 
    'baseline/inputs/model_info.json'
)

print(f"Uploaded baseline evaluation script to s3://{S3_BUCKET}/scripts/baseline_evaluation_script.py")
print(f"Uploaded model info to s3://{S3_BUCKET}/baseline/inputs/model_info.json")

## 6. Launch Distributed Baseline Evaluation Job

Now we'll set up and launch a SageMaker Processing job to perform the baseline evaluations. This job will run on a more powerful instance than our notebook, allowing us to evaluate larger models efficiently.

### Processing Job Configuration:
- **Instance Type**: We'll use the instance type specified in the workshop settings
- **Framework**: PyTorch for compatibility with the transformer models
- **Inputs**: The baseline evaluation script and model information
- **Outputs**: The baseline metrics and saved models

In [ ]:
# Define the instance type to use for baseline evaluationinstance_type = OPTIMIZATION_INSTANCE_TYPEprint(f"Using instance type: {instance_type} for baseline evaluation")# Create a SageMaker sessionsagemaker_session = sagemaker.Session()# Create a PyTorch processorprocessor = PyTorchProcessor(    framework_version="1.13.1",    py_version="py39",    role=SAGEMAKER_ROLE_ARN,    instance_type=instance_type,    instance_count=1,    base_job_name="baseline-evaluation",    sagemaker_session=sagemaker_session,    # Add required packages    dependencies=["transformers", "datasets"])# Define inputs and outputsinputs = [    ProcessingInput(        source=f's3://{S3_BUCKET}/scripts/baseline_evaluation_script.py',        destination='/opt/ml/processing/input/code/baseline_evaluation_script.py'    ),    ProcessingInput(        source=f's3://{S3_BUCKET}/baseline/inputs/model_info.json',        destination='/opt/ml/processing/input/data/model_info.json'    )]outputs = [    ProcessingOutput(        source='/opt/ml/processing/output',        destination=f's3://{S3_BUCKET}/baseline/outputs'    )]# Run the processing jobbaseline_job = processor.run(    code='/opt/ml/processing/input/code/baseline_evaluation_script.py',    inputs=inputs,    outputs=outputs,    arguments=[        '--model-info-path', '/opt/ml/processing/input/data/model_info.json',        '--output-dir', '/opt/ml/processing/output'    ])print(f"Launched baseline evaluation job: {baseline_job.job_name}")try:

## 7. Monitor Job Status

After launching the baseline evaluation job, we need to monitor its progress. SageMaker Processing jobs run asynchronously, so we'll periodically check the status until the job is complete.

### Job Status Lifecycle:
- **InProgress**: The job is currently running
- **Completed**: The job has successfully completed
- **Failed**: The job encountered an error and failed
- **Stopping**: The job is in the process of stopping
- **Stopped**: The job was manually stopped

We'll display the status and update it every 30 seconds until the job is complete.

In [ ]:
# Monitor job statusimport time# Create a SageMaker clientsagemaker_client = boto3.client('sagemaker')# Check job status every 30 secondsstatus = sagemaker_client.describe_processing_job(    ProcessingJobName=baseline_job.job_name)['ProcessingJobStatus']print(f"Job status: {status}")while status == 'InProgress':    time.sleep(30)    status = sagemaker_client.describe_processing_job(        ProcessingJobName=baseline_job.job_name    )['ProcessingJobStatus']    print(f"Job status: {status}")if status == 'Completed':    print("Baseline evaluation job completed successfully!")    print("Baseline evaluation job completed successfully!")else:    print(f"Job ended with status: {status}")

## 8. Collect Results

Once the job is complete, we'll download and analyze the baseline metrics.

In [ ]:
# Download baseline metrics
s3_client.download_file(
    S3_BUCKET,
    'baseline/outputs/baseline_metrics.json',
    'baseline_metrics.json'
)

# Load baseline metrics
with open('baseline_metrics.json', 'r') as f:
    baseline_metrics = json.load(f)

print(f"Downloaded baseline metrics for {len(baseline_metrics)} models")

## 9. Analyze Baseline Metrics

Now let's analyze the baseline metrics to understand the performance characteristics of our models.

In [ ]:
# Create a DataFrame with the metrics for better display
model_names = [metrics["model_name"] for metrics in baseline_metrics.values()]
model_sizes = [metrics["model_size"] for metrics in baseline_metrics.values()]
inference_times = [metrics["inference_time"] for metrics in baseline_metrics.values()]
num_parameters = [metrics["num_parameters"] for metrics in baseline_metrics.values()]

# Create a DataFrame
metrics_df = pd.DataFrame({
    "Model": model_names,
    "Size (MB)": model_sizes,
    "Inference Time (ms)": inference_times,
    "Parameters": num_parameters
})

# Display the metrics table
metrics_df

## 10. Visualize Baseline Metrics

Let's create some visualizations to better understand the baseline metrics.

In [ ]:
# Create visualizations
plt.figure(figsize=(15, 10))

# Model size plot
plt.subplot(2, 2, 1)
sns.barplot(x=metrics_df["Model"], y=metrics_df["Size (MB)"])
plt.title("Model Size (MB)")
plt.xticks(rotation=45, ha='right')
plt.tight_layout()

# Inference time plot
plt.subplot(2, 2, 2)
sns.barplot(x=metrics_df["Model"], y=metrics_df["Inference Time (ms)"])
plt.title("Inference Time (ms)")
plt.xticks(rotation=45, ha='right')
plt.tight_layout()

# Number of parameters plot
plt.subplot(2, 2, 3)
sns.barplot(x=metrics_df["Model"], y=metrics_df["Parameters"])
plt.title("Number of Parameters")
plt.xticks(rotation=45, ha='right')
plt.tight_layout()

# Size vs. Inference Time scatter plot
plt.subplot(2, 2, 4)
sns.scatterplot(x=metrics_df["Size (MB)"], y=metrics_df["Inference Time (ms)"])
plt.title("Size vs. Inference Time")
plt.xlabel("Size (MB)")
plt.ylabel("Inference Time (ms)")
for i, model in enumerate(metrics_df["Model"]):
    plt.annotate(model, (metrics_df["Size (MB)"][i], metrics_df["Inference Time (ms)"][i]))
plt.tight_layout()

plt.show()

## 11. Next Steps

Now that we've established baseline metrics for our models, we'll explore quantization techniques in the next notebook to reduce model size and improve inference speed.

### What We've Learned:
- How to use SageMaker Processing for distributed model evaluation
- How to measure key performance metrics for transformer models
- How model size relates to inference time
- The baseline performance characteristics of our models

### What's Next - Quantization:
Quantization is a technique that reduces the precision of the numbers used to represent a model's parameters. For example, converting 32-bit floating point numbers to 8-bit integers. This significantly reduces model size and can improve inference speed, often with minimal impact on accuracy.